# 3. Merge

Junta item + receita + bancada manual (`bench_recipes.json`) num único
DataFrame de ingredientes em formato *long*, pronto para as validações das
próximas etapas.


In [1]:
import json
from pathlib import Path

import pandas as pd

# Notebook lives in notebooks/etl/, so the repo root is two levels up.
REPO_ROOT = Path("../..").resolve()

ROOT = REPO_ROOT / "data/Pal/Content"
PAL = ROOT / "Pal"
ITEM_DT = PAL / "DataTable/Item/DT_ItemDataTable_Common.json"
RECIPE_DT = PAL / "DataTable/Item/DT_ItemRecipeDataTable_Common.json"
BUILDOBJECT_DT = PAL / "DataTable/MapObject/Building/DT_BuildObjectDataTable_Common.json"
BENCH_RECIPES = REPO_ROOT / "src/data/bench_recipes.json"
NAMES_DT_EN = ROOT / "L10N/en/Pal/DataTable/Text/DT_ItemNameText_Common.json"
NAMES_DT_PT_BR = ROOT / "L10N/pt-BR/Pal/DataTable/Text/DT_ItemNameText_Common.json"


def load_rows(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)[0]["Rows"]


In [2]:
items_raw = load_rows(ITEM_DT)
recipes_raw = load_rows(RECIPE_DT)
buildings_raw = load_rows(BUILDOBJECT_DT)

items_df = pd.DataFrame.from_dict(items_raw, orient="index")
recipes_df = pd.DataFrame.from_dict(recipes_raw, orient="index")
buildings_df = pd.DataFrame.from_dict(buildings_raw, orient="index")

print(f"items: {items_df.shape}, recipes: {recipes_df.shape}, buildings: {buildings_df.shape}")


items: (2466, 53), recipes: (1414, 20), buildings: (498, 32)


In [3]:
# Padroniza a string sentinela "None" (usada pela UE pra ausência de valor)
# para o None real do Python, e monta um resolvedor de id case-insensitive -
# ver 02_limpeza.ipynb pro raciocínio completo por trás disso.
items_clean = items_df.replace("None", None)
recipes_clean = recipes_df.replace("None", None)
buildings_clean = buildings_df.replace("None", None)

items_by_lower = {item_id.lower(): item_id for item_id in items_clean.index}


def resolve_item_id(raw_id):
    # pandas' .replace("None", None) upcasts these cells to NaN (a float),
    # not Python None - "is None" or plain truthiness checks silently miss
    # it and .lower() blows up on a float. pd.isna() catches both.
    if pd.isna(raw_id):
        return None
    if raw_id in items_clean.index:
        return raw_id
    return items_by_lower.get(raw_id.lower())


## Recipes em formato long

Cada `MaterialX_Id`/`MaterialX_Count` vira uma linha própria, com a
quantidade **crua** de material e o `product_count` (rendimento por craft) -
mesma regra de `scripts/crafting_graph.py`: não normalizamos por unidade;
uma receita consome `material_count` e produz `product_count` unidades.


In [4]:
rows = []
for product_id, recipe in recipes_clean.iterrows():
    product_count = recipe["Product_Count"] or 1
    for i in range(1, 6):
        material_id = recipe[f"Material{i}_Id"]
        material_count = recipe[f"Material{i}_Count"] or 0
        if pd.isna(material_id) or material_count == 0:
            continue
        rows.append({
            "item_id": product_id,
            "ingredient_id_raw": material_id,
            "ingredient_id": resolve_item_id(material_id),
            "material_count": material_count,
            "product_count": product_count,
        })

ingredients_long = pd.DataFrame(rows)
print(ingredients_long.shape)
ingredients_long.head()


(3937, 5)


,item_id,ingredient_id_raw,ingredient_id,material_count,product_count
0,Money,CopperIngot,CopperIngot,30,20000
1,PalSphere,Pal_crystal_S,Pal_crystal_S,1,1
2,PalSphere_Mega,Pal_crystal_S,Pal_crystal_S,1,1
3,PalSphere_Mega,CopperIngot,CopperIngot,1,1
4,PalSphere_Mega,Wood,Wood,3,1


## Reachable set: craftáveis + tudo que é usado como ingrediente


In [5]:
reachable = set(recipes_clean.index) | set(ingredients_long["ingredient_id"].dropna())
print(f"{len(reachable)} ids no reachable set")


1622 ids no reachable set


## Merge final: item + workbench (bancada manual)


In [6]:
bench_recipes = json.load(open(BENCH_RECIPES, encoding="utf-8"))["items"]
workbench_series = pd.Series(
    {k: v["primaryBench"] for k, v in bench_recipes.items()}, name="workbench"
)

items_merged = items_clean.reindex(sorted(reachable)).join(workbench_series, how="left")
items_merged["is_base"] = ~items_merged.index.isin(recipes_clean.index)

print(f"{len(items_merged)} linhas no DataFrame mergeado")
items_merged[["TypeA", "TypeB", "Price", "Rarity", "workbench", "is_base"]].head(10)


1622 linhas no DataFrame mergeado


,TypeA,TypeB,Price,Rarity,workbench,is_base
AIcore,EPalItemTypeA::Material,EPalItemTypeB::MaterialProccessing,252940.0,0.0,Factory_Hard_04,False
Accessory_AT_1,EPalItemTypeA::Accessory,EPalItemTypeB::Accessory,16500.0,2.0,Factory_Hard_01,False
Accessory_AirDash1,EPalItemTypeA::Accessory,EPalItemTypeB::Accessory,54120.0,2.0,Factory_Hard_01,False
Accessory_AirDash2,EPalItemTypeA::Accessory,EPalItemTypeB::Accessory,153120.0,3.0,Factory_Hard_02,False
Accessory_AirDash3,EPalItemTypeA::Accessory,EPalItemTypeB::Accessory,306600.0,4.0,Factory_Hard_02,False
Accessory_AquaResist_1,EPalItemTypeA::Accessory,EPalItemTypeB::Accessory,23880.0,2.0,Factory_Hard_01,False
Accessory_Avoid_1,EPalItemTypeA::Accessory,EPalItemTypeB::Accessory,121860.0,3.0,Factory_Hard_02,False
Accessory_ColdIce_1,EPalItemTypeA::Accessory,EPalItemTypeB::Accessory,158040.0,3.0,Factory_Hard_02,False
Accessory_CoolResist_1,EPalItemTypeA::Accessory,EPalItemTypeB::Accessory,21960.0,2.0,Factory_Hard_01,False
Accessory_DFHP_1,EPalItemTypeA::Accessory,EPalItemTypeB::Accessory,80700.0,3.0,Factory_Hard_02,False


## Preview: ingredientes agregados de um item específico


In [7]:
sample_item_id = recipes_clean.index[0]
sample = ingredients_long[ingredients_long["item_id"] == sample_item_id]
print(f"Ingredientes de {sample_item_id}:")
sample[["ingredient_id", "material_count", "product_count"]]


Ingredientes de Money:


,ingredient_id,material_count,product_count
0,CopperIngot,30,20000
